# Country Mentions Heatmap Analysis

## Overview
This notebook analyses country mentions in UN briefing documents and generates an interactive Bokeh heatmap visualisation showing mention frequency by country and month.

## Features
- Scrapes/loads country data from Wikipedia or CSV
- Extracts briefing sections from SQLite database
- Counts country mentions using regex pattern matching
- Generates interactive Bokeh heatmap with log-scale coloring
- Exports data to CSV for further analysis

## Imports

In [1]:
import sqlite3
import re
from pathlib import Path
from typing import Dict

import pandas as pd
import numpy as np
import curl_cffi
from bs4 import BeautifulSoup
from bokeh.plotting import figure, output_file, save
from bokeh.models import HoverTool, ColumnDataSource, ColorBar, LinearColorMapper
from bokeh.palettes import Viridis256
from bokeh.transform import transform
from loguru import logger

## Function Definitions

In [2]:
def get_countries_from_wikipedia() -> pd.DataFrame:
    """
    Scrape country data from Wikipedia.
    
    Returns:
        DataFrame with columns: code, name, alternative_names
    """
    url = "https://en.wikipedia.org/wiki/List_of_alternative_country_names"
    response = curl_cffi.get(url, impersonate="chrome")
    soup = BeautifulSoup(response.text, "html.parser")
    tables = soup.find_all(class_="wikitable")

    data = []
    for table in tables:
        for row in table.find_all("tr")[1:]:  # Skip header row
            cell_data = row.find_all("td")
            if len(cell_data) == 3:
                code, name, names = cell_data
                country_name = name.find("a")["title"].strip()
                other_names = "|".join(
                    [x.text.strip() for x in names.find_all("b")]
                    + [x.text.strip() for x in names.find_all("i")]
                )
                data.append((code.text.strip(), country_name, other_names))

    return pd.DataFrame(data, columns=["code", "name", "alternative_names"])

In [3]:
def get_countries_data(csv_path: Path | str) -> pd.DataFrame:
    """Load country data from CSV reference file."""
    return pd.read_csv(csv_path)

In [4]:
def get_sections_from_db(db_path: Path | str) -> pd.DataFrame:
    """Extract briefing sections from SQLite database."""
    conn = sqlite3.connect(db_path)

    query = """
    SELECT
        s.section_id,
        s.briefing_id,
        s.text,
        b.briefing_date
    FROM section s
    JOIN briefing b ON s.briefing_id = b.briefing_id
    WHERE s.text IS NOT NULL AND b.briefing_date IS NOT NULL
    """

    sections_df = pd.read_sql_query(query, conn)
    conn.close()

    # Parse date and extract month
    sections_df['briefing_date'] = pd.to_datetime(sections_df['briefing_date'])
    sections_df['month'] = sections_df['briefing_date'].dt.to_period('M')

    return sections_df[['section_id', 'text', 'month']]

In [5]:
def clean_alternative_names(alt_names_str: str) -> str:
    """Remove problematic abbreviations from alternative names."""
    if pd.isna(alt_names_str):
        return ""
    
    alt_names = alt_names_str.split("|")
    problematic = {'png', 'and', 'or', 'usa'}
    alt_names = [
        name for name in alt_names 
        if name.strip().lower() not in problematic
    ]
    
    return "|".join(alt_names)

In [6]:
def build_search_dict(country_df: pd.DataFrame) -> Dict[str, str]:
    """Build mapping of search terms to country codes."""
    search_dict = {}

    for _, row in country_df.iterrows():
        code = row['code'].strip() if pd.notna(row['code']) else ""
        name = row['name'].strip() if pd.notna(row['name']) else ""
        alt_names = row['alternative_names'].split("|") if pd.notna(row['alternative_names']) else []

        country_key = code if code else name

        if name:
            search_dict[name.lower()] = country_key
        
        for alt in alt_names:
            alt = alt.strip().lower()
            if alt and any(c.isalnum() for c in alt):
                search_dict[alt] = country_key

    return search_dict

In [7]:
def compile_search_pattern(search_dict: Dict[str, str]) -> re.Pattern:
    """Compile regex pattern for efficient multi-term matching."""
    pattern = r'\b(' + '|'.join(re.escape(term) for term in search_dict.keys()) + r')\b'
    return re.compile(pattern, re.IGNORECASE)

In [8]:
def count_mentions_in_section(
    section_text: str,
    search_dict: Dict[str, str],
    compiled_regex: re.Pattern
) -> Dict[str, int]:
    """Count country mentions in a single section of text."""
    counts = {}

    for match in compiled_regex.finditer(section_text):
        matched_term = match.group(1).lower()
        country_key = search_dict[matched_term]
        counts[country_key] = counts.get(country_key, 0) + 1

    return counts

In [9]:
def process_sections_to_heatmap(
    sections_df: pd.DataFrame,
    country_df: pd.DataFrame
) -> pd.DataFrame:
    """Process all sections and generate heatmap data."""
    search_dict = build_search_dict(country_df)
    compiled_regex = compile_search_pattern(search_dict)

    all_country_counts = {}

    for _, row in sections_df.iterrows():
        section_text = row['text']
        section_month = row['month']

        counts = count_mentions_in_section(section_text, search_dict, compiled_regex)

        for country_code, count in counts.items():
            key = (section_month, country_code)
            all_country_counts[key] = all_country_counts.get(key, 0) + count

    heatmap_data = [
        {
            'month': str(month),
            'country_code': code,
            'count': count
        }
        for (month, code), count in all_country_counts.items()
    ]

    return pd.DataFrame(heatmap_data)

In [10]:
def create_bokeh_heatmap(
    heatmap_df: pd.DataFrame,
    output_path: str = 'heatmap.html',
    top_n: int = 25
) -> None:
    """Create interactive Bokeh heatmap visualization."""
    top_countries = heatmap_df.groupby('country_code')['count'].sum().nlargest(top_n).index
    heatmap_df = heatmap_df[heatmap_df['country_code'].isin(top_countries)].copy()
    heatmap_df = heatmap_df.sort_values(['month', 'country_code'])

    heatmap_df['count_log'] = np.log1p(heatmap_df['count'])
    max_count_log = heatmap_df['count_log'].max()

    color_mapper = LinearColorMapper(palette=Viridis256, low=0, high=max_count_log)
    source = ColumnDataSource(heatmap_df)

    p = figure(
        title=f"Country Mentions Heatmap - Top {top_n} Countries (log scale)",
        x_axis_label='Month',
        y_axis_label='Country',
        x_range=sorted(heatmap_df['month'].unique()),
        y_range=sorted(heatmap_df['country_code'].unique()),
        width=1200,
        height=600 + (top_n * 15),
        toolbar_location='right',
        tools='pan,wheel_zoom,box_zoom,reset,save'
    )

    p.rect(
        x='month', y='country_code', width=0.9, height=0.9, source=source,
        fill_color=transform('count_log', color_mapper), line_color='white', line_width=1
    )

    hover = HoverTool(tooltips=[('Month', '@month'), ('Country', '@country_code'), ('Mentions', '@count')])
    p.add_tools(hover)

    color_bar = ColorBar(color_mapper=color_mapper, label_standoff=12, title='Mentions (log)')
    p.add_layout(color_bar, 'right')

    p.yaxis.major_label_text_font_size = "10pt"
    p.xaxis.major_label_orientation = 0.785
    p.grid.visible = False

    output_file(output_path)
    save(p)
    logger.info(f"Heatmap saved to {output_path}")

## Configuration

In [11]:
def get_repo_root() -> Path:
    """Find repo root by looking for .git directory."""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / ".git").exists():
            return parent
    return current

REPO_ROOT = get_repo_root()
DB_PATH = REPO_ROOT / "data" / "processed" / "database" / "briefings.db"
COUNTRY_CSV_PATH = REPO_ROOT / "data" / "countries_reference.csv"
OUTPUT_DIR = REPO_ROOT / "data" / "output"

# Create output directory if needed
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Database path: {DB_PATH}")
logger.info(f"Database exists: {DB_PATH.exists()}")
logger.info(f"Output directory: {OUTPUT_DIR}")

2026-09-24 13:12:45.992 | INFO     | __main__:<module>:17 - Database path: C:\Users\danie\PycharmProjects\correlaid-mentoring-project\data\processed\database\briefings.db
2026-09-24 13:12:45.994 | INFO     | __main__:<module>:18 - Database exists: True
2026-09-24 13:12:45.995 | INFO     | __main__:<module>:19 - Output directory: C:\Users\danie\PycharmProjects\correlaid-mentoring-project\data\output


## Main Pipeline Execution

In [12]:
logger.info("\n" + "="*70)
logger.info("COUNTRY MENTIONS HEATMAP ANALYSIS - ISSUE #47")
logger.info("="*70 + "\n")

logger.info("Step 1: Loading countries...")
if COUNTRY_CSV_PATH.exists():
    country_df = get_countries_data(COUNTRY_CSV_PATH)
    logger.info(f"✓ Loaded {len(country_df)} countries from CSV")
else:
    logger.info("CSV not found, scraping from Wikipedia...")
    country_df = get_countries_from_wikipedia()
    country_df.to_csv(COUNTRY_CSV_PATH, index=False)
    logger.info(f"✓ Loaded {len(country_df)} countries and saved to CSV")

2026-09-24 13:12:46.022 | INFO     | __main__:<module>:1 - 
2026-09-24 13:12:46.024 | INFO     | __main__:<module>:2 - COUNTRY MENTIONS HEATMAP ANALYSIS - ISSUE #47
2026-09-24 13:12:46.024 | INFO     | __main__:<module>:3 - ======================================================================

2026-09-24 13:12:46.025 | INFO     | __main__:<module>:5 - Step 1: Loading countries...
2026-09-24 13:12:46.040 | INFO     | __main__:<module>:8 - ✓ Loaded 200 countries from CSV


In [13]:
logger.info("\nStep 2: Cleaning alternative names...")
country_df['alternative_names'] = country_df['alternative_names'].apply(clean_alternative_names)
logger.info("✓ Done")

2026-09-24 13:12:46.057 | INFO     | __main__:<module>:1 - 
Step 2: Cleaning alternative names...
2026-09-24 13:12:46.060 | INFO     | __main__:<module>:3 - ✓ Done


In [14]:
logger.info("\nStep 3: Extracting sections from database...")
sections_df = get_sections_from_db(DB_PATH)
logger.info(f"✓ Loaded {len(sections_df)} sections")

2026-09-24 13:12:46.083 | INFO     | __main__:<module>:1 - 
Step 3: Extracting sections from database...
2026-09-24 13:12:46.227 | INFO     | __main__:<module>:3 - ✓ Loaded 14174 sections


In [15]:
logger.info("\nStep 4: Processing country mentions...")
heatmap_df = process_sections_to_heatmap(sections_df, country_df)
logger.info(f"✓ Generated heatmap with {len(heatmap_df)} data points")

2026-09-24 13:12:46.242 | INFO     | __main__:<module>:1 - 
Step 4: Processing country mentions...
2026-09-24 13:13:48.786 | INFO     | __main__:<module>:3 - ✓ Generated heatmap with 5766 data points


In [16]:
logger.info("\nStep 5: Exporting data...")
csv_path = OUTPUT_DIR / 'heatmap_data.csv'
heatmap_df.to_csv(csv_path, index=False)
logger.info(f"✓ Exported to {csv_path}")

2026-09-24 13:13:48.802 | INFO     | __main__:<module>:1 - 
Step 5: Exporting data...
2026-09-24 13:13:48.813 | INFO     | __main__:<module>:4 - ✓ Exported to C:\Users\danie\PycharmProjects\correlaid-mentoring-project\data\output\heatmap_data.csv


In [17]:
logger.info("\nStep 6: Creating Bokeh visualization...")
html_path = OUTPUT_DIR / 'heatmap.html'
create_bokeh_heatmap(heatmap_df, output_path=str(html_path), top_n=25)
logger.info("✓ Analysis complete!")

2026-09-24 13:13:48.833 | INFO     | __main__:<module>:1 - 
Step 6: Creating Bokeh visualization...
2026-09-24 13:13:49.023 | INFO     | __main__:create_bokeh_heatmap:46 - Heatmap saved to C:\Users\danie\PycharmProjects\correlaid-mentoring-project\data\output\heatmap.html
2026-09-24 13:13:49.025 | INFO     | __main__:<module>:4 - ✓ Analysis complete!


## Analysis & Diagnostics

In [18]:
# Data summary
logger.info("\n" + "="*70)
logger.info("DATA SUMMARY")
logger.info("="*70)

logger.info("\nDistribution of country mentions:")
print(heatmap_df.groupby('country_code')['count'].sum().describe())

total_mentions = heatmap_df['count'].sum()
top_25_countries = heatmap_df.groupby('country_code')['count'].sum().nlargest(25).index
top_25_mentions = heatmap_df[heatmap_df['country_code'].isin(top_25_countries)]['count'].sum()
pct = (top_25_mentions / total_mentions) * 100

logger.info(f"\nTop 25 countries: {top_25_mentions:,.0f} mentions ({pct:.1f}% of total)")
logger.info(f"Remaining {len(heatmap_df['country_code'].unique()) - 25} countries: "
      f"{total_mentions - top_25_mentions:,.0f} mentions ({100-pct:.1f}%)")

logger.info("\nTop 10 countries by mentions:")
top_countries = heatmap_df.groupby('country_code')['count'].sum().nlargest(10)
print(top_countries)
logger.info("="*70)

2026-09-24 13:13:49.042 | INFO     | __main__:<module>:2 - 
2026-09-24 13:13:49.043 | INFO     | __main__:<module>:3 - DATA SUMMARY
2026-09-24 13:13:49.043 | INFO     | __main__:<module>:4 - ======================================================================
2026-09-24 13:13:49.045 | INFO     | __main__:<module>:6 - 
Distribution of country mentions:
2026-09-24 13:13:49.057 | INFO     | __main__:<module>:14 - 
Top 25 countries: 27,148 mentions (72.3% of total)
2026-09-24 13:13:49.059 | INFO     | __main__:<module>:15 - Remaining 171 countries: 10,415 mentions (27.7%)
2026-09-24 13:13:49.059 | INFO     | __main__:<module>:18 - 
Top 10 countries by mentions:
2026-09-24 13:13:49.065 | INFO     | __main__:<module>:21 - ======================================================================


count     196.000000
mean      191.647959
std       461.066660
min         1.000000
25%        22.750000
50%        50.500000
75%       116.000000
max      3210.000000
Name: count, dtype: float64
country_code
SDN    3210
USA    2869
ZAF    2677
UKR    2332
LBN    1785
SYR    1593
COD    1291
AFG    1183
YEM    1138
HTI    1102
Name: count, dtype: int64


## Country-Specific Analysis

In [21]:
# Analyze a specific country
country_code = 'USA'

logger.info(f"\nAnalyzing mentions of {country_code}:")
logger.info("-" * 50)

country_sections = sections_df[sections_df['text'].str.contains(country_code, case=False, na=False)]

if len(country_sections) > 0:
    total_mentions = heatmap_df[heatmap_df['country_code'] == country_code]['count'].sum()
    logger.info(f"Total mentions: {total_mentions}")
    logger.info(f"Sections containing term: {len(country_sections)}")
    logger.info(f"Average mentions per section: {total_mentions / len(country_sections):.1f}")
    
    by_month = heatmap_df[heatmap_df['country_code'] == country_code].sort_values('count', ascending=False)
    logger.info("\nTop 5 months:")
    print(by_month.head(5)[['month', 'count']])
else:
    logger.info(f"No sections found mentioning {country_code}")

2026-09-24 13:15:14.671 | INFO     | __main__:<module>:4 - 
Analyzing mentions of USA:
2026-09-24 13:15:14.672 | INFO     | __main__:<module>:5 - --------------------------------------------------
2026-09-24 13:15:14.703 | INFO     | __main__:<module>:11 - Total mentions: 2869
2026-09-24 13:15:14.704 | INFO     | __main__:<module>:12 - Sections containing term: 1159
2026-09-24 13:15:14.705 | INFO     | __main__:<module>:13 - Average mentions per section: 2.5
2026-09-24 13:15:14.709 | INFO     | __main__:<module>:16 - 
Top 5 months:


        month  count
4377  2025-04     82
5333  2026-04     77
5485  2026-06     76
4772  2025-09     76
5262  2026-03     74
